# 03.03 LoRA 配置、训练与推理

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 03.02，模型已下载、数据已预处理</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">配置 LoRA 参数，启动训练，对比微调前后推理效果</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">LoRA 原理与配置 → 模型训练 → 微调后推理对比</td></tr>
</table>

> ⏱️ 本节训练耗时约 15-30 分钟。

## 第一部分：LoRA 原理与参数配置


## 1. 环境与数据准备（独立运行时需要）

本notebook用到模型加载、LoRA配置、训练、推理，需要先完成依赖导入、模型加载和数据预处理。运行下方cell一次性准备完成。

> 💡 如果已按顺序从 03.02 运行过来，可跳过本cell。

In [ ]:
# ===== 环境与数据准备（让本notebook可独立运行）=====
# 本cell导入全部依赖 + 加载模型 + 预处理数据，完成后可直接进入下方 LoRA 配置/训练/推理
import os, glob, pandas as pd
import torch
import torch_npu  # noqa: F401
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          DataCollatorForSeq2Seq, TrainingArguments, Trainer,
                          GenerationConfig)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

MODEL_PATH = './src/DeepSeek-R1-Distill-Qwen-1.5B'

# 1. 实例化 tokenizer（假设模型已下载，详见 03.02）
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False, trust_remote_code=True)

# 2. 数据预处理（假设数据已下载，详见 03.02）
df = pd.read_json('./src/huanhuan.json')
ds = Dataset.from_pandas(df)

def process_func(example):
    # 用模型原生 apply_chat_template 生成 prompt（DeepSeek 模板，非 ChatML）
    # 详见 03.02 的说明：DeepSeek-R1-Distill 词表无 <|im_start|>，必须用原生模板
    MAX_LENGTH = 384
    instruction = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "现在你要扮演皇帝身边的女人--甄嬛"},
            {"role": "user", "content": example['instruction'] + example['input']},
        ],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
    )
    response = tokenizer(example['output'], add_special_tokens=False)
    input_ids = instruction['input_ids'] + response['input_ids'] + [tokenizer.pad_token_id]
    attention_mask = instruction['attention_mask'] + response['attention_mask'] + [1]
    labels = [-100] * len(instruction['input_ids']) + response['input_ids'] + [tokenizer.pad_token_id]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

tokenized_id = ds.map(process_func, remove_columns=ds.column_names)
print('✅ 环境与数据准备完成：tokenizer、tokenized_id 就绪')
print(f'   数据集: {len(tokenized_id)} 条样本')

In [ ]:
# 加载基础模型
# torch_dtype=torch.bfloat16：用 bfloat16 精度加载，节省一半显存
# device_map：自动把模型放到可用设备（NPU/CPU）
model = AutoModelForCausalLM.from_pretrained('./src/DeepSeek-R1-Distill-Qwen-1.5B',
                                             torch_dtype=torch.bfloat16, device_map='npu')

# 开启梯度检查点时，必须调用此方法（让输入张量能够反传梯度）
model.enable_input_require_grads()


> 💡 **关于 NPU 设备**：代码中 <code>model.to('npu')</code> 把模型从 host 搬到昇腾 NPU；输入张量同样 <code>.to('npu')</code>。<code>torch_npu</code> 扩展让 PyTorch 原生支持昇腾，API 与 CUDA 版本几乎一致（<code>torch.npu</code> 对应 <code>torch.cuda</code>）。

In [ ]:
# 本cell用到前序cell的 model、tokenizer、torch（请按顺序运行）
# host to device：把模型搬到 NPU（如已用 device_map 放置则可省略）
model = model.to('npu')

prompt = "你是谁？"
inputs = tokenizer.apply_chat_template(
    [{"role": "system", "content": "现在你要扮演皇帝身边的女人--甄嬛"},
     {"role": "user", "content": prompt}],
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True
).to('npu')

# 贪心解码（do_sample=False）：结果可复现，便于对比微调前后效果
gen_kwargs = {"max_new_tokens": 512, "do_sample": False}
with torch.no_grad():
    outputs = model.generate(**inputs, **gen_kwargs)
    outputs = outputs[:, inputs['input_ids'].shape[1]:]   # 去掉输入部分，只取新生成的
    print("【微调前】模型回答：")
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))


观察输出：基础模型（DeepSeek-R1 蒸馏版）会输出冗长的 `<think>...</think>` 思维链，且人设不稳，不会以"甄嬛"的口吻回答。这正是我们要用微调改善的。

## 2. 配置 LoRA 参数

现在为模型配置 LoRA。`LoraConfig` 中的关键参数：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">参数</th><th align="left">本实验取值</th><th align="left">含义</th></tr>
<tr><td align="left"><code>task_type</code></td><td align="left">CAUSAL_LM</td><td align="left">任务类型：因果语言模型（自回归生成）</td></tr>
<tr><td align="left"><code>r</code></td><td align="left">8</td><td align="left">低秩矩阵的秩，控制可训练参数规模。r 越大表达能力越强但参数越多</td></tr>
<tr><td align="left"><code>lora_alpha</code></td><td align="left">32</td><td align="left">缩放因子，LoRA 分支的实际缩放为 <code>alpha/r = 4</code></td></tr>
<tr><td align="left"><code>target_modules</code></td><td align="left">7 个投影层</td><td align="left">注入 LoRA 的层（见下方说明）</td></tr>
<tr><td align="left"><code>lora_dropout</code></td><td align="left">0.1</td><td align="left">LoRA 分支的 Dropout 比例，防止过拟合</td></tr>
<tr><td align="left"><code>inference_mode</code></td><td align="left">False</td><td align="left">False 表示训练模式（需梯度）；True 表示仅推理</td></tr>
</table>

**target_modules 选择的 7 个层**：覆盖了 Qwen2 的全部 attention 投影（`q_proj/k_proj/v_proj/o_proj`）和 MLP 投影（`gate_proj/up_proj/down_proj`）。这是较为激进的配置，效果通常更好但参数略多；若显存紧张可只保留 attention 的 4 层。

In [ ]:
# 配置 LoRA（TaskType 来自 peft 库，PyTorch 通用）
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    inference_mode=False,   # 训练模式
    r=8,                    # LoRA 秩
    lora_alpha=32,          # 缩放因子，实际缩放 = alpha / r
    lora_dropout=0.1        # Dropout 比例
)
config


## 3. 观察模型结构变化

调用 `get_peft_model` 把 LoRA 分支"挂"到模型上。对比前后结构：

In [ ]:
print("=" * 30 + " 挂载 LoRA 前 " + "=" * 30)
print("Model without LoRA:\n", model)

# 根据上述 lora 配置，为模型添加 lora 部分
model = get_peft_model(model, config)

print("=" * 30 + " 挂载 LoRA 后 " + "=" * 30)
print("Model with LoRA:\n", model)

# 输出需要训练的参数比例
print("\n" + "=" * 30 + " 可训练参数统计 " + "=" * 30)
model.print_trainable_parameters()


### 结构变化解读

挂载 LoRA 之前，`Qwen2Attention`、`Qwen2MLP` 内含 `q_proj`、`gate_proj` 等较大的 `Linear` 层。挂载之后，这些模块都被包装为 `lora.Linear`：

- `base_layer`：原始 `Linear`（如 `1536 -> 1536`），参数被**冻结**
- `lora_A` / `lora_B`：新增的低秩投影，例如 `Linear(1536 -> 8)` 与 `Linear(8 -> 1536)`
- `lora_dropout`：分支上的 Dropout

最终可训练参数仅 **9,232,384**，占全部参数（约 17.86 亿）的 **0.5168%**。

LoRA 配置完成，下面进入模型训练与微调后推理。

---

## 本节练习

**练习 1（选择）**：LoRA 微调相比全量微调，最大的优势是什么？
- A. 训练出的模型更准确
- B. 可训练参数大幅减少，节省显存和存储
- C. 训练速度一定更快
- D. 不需要任何原始模型权重

**练习 2（填空）**：LoRA 的数学形式为 `W' = W + ______`，其中被冻结的矩阵是 ______，新增的低秩矩阵是 ______ 和 ______。

**练习 3（简答）**：本实验的 `lora_alpha=32`、`r=8`，那么 LoRA 分支的实际缩放系数是多少？为什么需要这个缩放？

**练习 4（代码）**：如果显存紧张，希望减少可训练参数，应该如何修改 `target_modules`？请写出只对 attention 层（4 个投影）注入 LoRA 的配置代码。

> 💡 参考答案见下方 code cell。


In [ ]:
# 查看本节练习答案
!cat ./answer/03.03_lora_train_infer/answers_lora.txt



---

## 第二部分：模型训练与微调后推理


In [ ]:
# 定义训练超参数
args = TrainingArguments(
    output_dir="./output_1.5bf/Qwen2.5_instruct_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=5,
    logging_steps=10,
    num_train_epochs=3,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
)


## 1. 启动训练

`Trainer` 是 transformers 提供的高层训练封装，自动处理：迭代数据、前向、反向、梯度更新、日志、checkpoint 保存。我们只需把模型、超参、数据集、`data_collator` 传进去：

- `data_collator=DataCollatorForSeq2Seq`：自动对同一 batch 内长短不一的样本做 padding，使它们能拼成张量；

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_id,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

# 启动训练（耗时 15-30 分钟）
trainer.train()


训练过程中你会看到 loss 逐步下降。训练完成后，`output_dir` 下会出现多个 `checkpoint-*` 子目录，每个目录里保存了对应步数的 LoRA 权重（`adapter_model.safetensors` 等）。

## 2. 微调后推理：对比效果

现在重新加载基础模型，并挂载刚训练得到的 LoRA 权重，验证微调效果。

> 💡 下方代码会**自动选取 `output_dir` 中最新的 checkpoint**，无需手动改路径。若你调整了训练超参，checkpoint 编号会随之变化，自动选取逻辑依然有效。

> ⚠️ 推理结束后可能打印一段 `[ERROR] TBE Subprocess[task_distribute] ... EOFError`，这是昇腾算子编译子进程在推理结束后的清理报错，**不影响推理结果**（模型回答已正常输出），可忽略。

In [ ]:
import os, glob
import torch
import torch_npu  # noqa: F401
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

model_path = './src/DeepSeek-R1-Distill-Qwen-1.5B'
output_dir = './output_1.5bf/Qwen2.5_instruct_lora'

# 自动选取最新 checkpoint（无需手动改路径）
ckpts = sorted(
    glob.glob(os.path.join(output_dir, 'checkpoint-*')),
    key=lambda p: int(os.path.basename(p).rsplit('-', 1)[1])
)
lora_path = ckpts[-1]
print('加载 LoRA 权重：', lora_path)

# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# 加载基础模型（注意是 .eval() 推理模式）
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype=torch.bfloat16, trust_remote_code=True
).eval()

# 挂载训练好的 LoRA 权重
model = PeftModel.from_pretrained(model, model_id=lora_path)

# host to device
model = model.to('npu')

# 用微调前同一个问题测试
prompt = "你是谁？"
inputs = tokenizer.apply_chat_template(
    [{"role": "system", "content": "现在你要扮演皇帝身边的女人--甄嬛"},
     {"role": "user", "content": prompt}],
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt",
    return_dict=True
).to('npu')

# 贪心解码（do_sample=False）：结果可复现，便于对比微调前后效果
gen_kwargs = {"max_new_tokens": 512, "do_sample": False}
with torch.no_grad():
    outputs = model.generate(**inputs, **gen_kwargs)
    outputs = outputs[:, inputs['input_ids'].shape[1]:]
print("【微调后】模型回答：")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### 微调前后对比

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">阶段</th><th align="left">对"你是谁？"的回答</th></tr>
<tr><td align="left"><b>微调前</b></td><td align="left">输出冗长的 <code>&lt;think&gt;...&lt;/think&gt;</code> 思维链，人设不稳，不按甄嬛口吻回答</td></tr>
<tr><td align="left"><b>微调后</b></td><td align="left">直接输出符合甄嬛人设的精炼回答，如「臣妾是甄嬛，家父是大理寺少卿甄远道。」</td></tr>
</table>

可以看到，仅用 0.52% 的参数进行 LoRA 微调，就让模型成功学会了甄嬛的角色设定。

---

## 本节练习

**练习 1（选择）**：`gradient_accumulation_steps=5` 且 `per_device_train_batch_size=4` 时，等效 batch_size 是多少？
- A. 4
- B. 5
- C. 20
- D. 9

**练习 2（填空）**：推理时为了让模型只生成新内容（不重复输入），需要用 `outputs[:, ______:]` 切片去掉输入部分。

**练习 3（简答）**：为什么 LoRA 微调的学习率（1e-4）通常比全量微调（如 2e-5）更大？

> 💡 参考答案见下方 code cell。


In [ ]:
# 查看本节练习答案
!cat ./answer/03.03_lora_train_infer/answers_train.txt
